# Evaluating the Spread of AI-Generated Synthetic Media on X

This notebook processes Community Notes data to identify and analyze tweets containing misleading synthetic media.

## 1. Setup and Configuration

In [ ]:
import os
import re
import time
import pickle
import urllib.request
from datetime import datetime

import pandas as pd
from google.colab import drive
from selenium import webdriver
from selenium.webdriver.common.by import By

# Configuration
DATA_PATH = "path-to-notes-tsv-file"
OUTPUT_DIR_IMAGES = "path-to-images-folder"
OUTPUT_FILE_INTERMEDIATE = "path-to-intermediate-file.csv"
OUTPUT_FILE_USERS = "path-to-users-file.csv"
OUTPUT_FILE_FINAL = "path-to-clean-file.csv"

DATE_START = "2022-11-01"
DATE_END = "2023-09-30"
SCRAPE_DELAY_SECONDS = 15
BATCH_SIZE = 40
BATCH_PAUSE_SECONDS = 600

## 2. Load Community Notes Data

In [ ]:
drive.mount('/content/drive')

def load_community_notes(path, start_date, end_date):
    """Load and filter Community Notes data to the specified date range."""
    df = pd.read_csv(path, delimiter='\t')
    df["date"] = pd.to_datetime(df['createdAtMillis'], unit='ms').dt.strftime('%d/%m/%Y')
    
    dates = pd.to_datetime(df['date'], format='%d/%m/%Y')
    mask = (dates > start_date) & (dates < end_date)
    return df[mask]

community_notes_data = load_community_notes(DATA_PATH, DATE_START, DATE_END)

## 3. Filter Notes by Synthetic Media Keywords

This filters for instances of AI-generated visual content that could be misleading (deepfakes, AI-generated images, etc.), while excluding explicitly labeled AI art such as #aiart content.

In [ ]:
SYNTHETIC_MEDIA_PATTERNS = [
    r"ai-generated(?:\s+image|\s+video|\s+art|\s+photo|\s+deepfake)",
    r"ai generated(?:\s+image|\s+video|\s+art|\s+photo|\s+deepfake)",
    r"ai(?:\s+image|\s+video|\s+art|\s+photo|\s+deepfake)",
    r"ai-(?:\s+image|\s+video|\s+art|\s+photo|\s+deepfake)",
    r"generated\s+(?:with|by)\s+(?:ai|artificial intelligence)",
    r"midjourney",
    r"stable diffusion",
    r"dall-e",
    r"deepfake",
    r"deep fake",
    r"deepfaked",
]

def filter_by_patterns(df, patterns):
    """Filter dataframe rows where 'summary' matches any of the given patterns."""
    combined_pattern = '|'.join(patterns)
    return df[df['summary'].str.contains(combined_pattern, case=False, na=False)]

visual_notes = filter_by_patterns(community_notes_data, SYNTHETIC_MEDIA_PATTERNS)
non_visual_notes = community_notes_data.drop(visual_notes.index)

community_notes = {
    'visual': visual_notes,
    'no_visual': non_visual_notes
}

with open('community-notes-filtered.pkl', 'wb') as f:
    pickle.dump(community_notes, f)

## 4. Tweet Data Collection

Uses Selenium to scrape tweet data including usernames, follower counts, engagement metrics, and embedded images.

### 4.1 Browser Setup

In [ ]:
def create_chrome_driver():
    """Create a headless Chrome WebDriver instance."""
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    return webdriver.Chrome(options=options)

### 4.2 Main Collection Pipeline

In [ ]:
ENGAGEMENT_PATTERNS = {
    'views': re.compile(r'([\d,.]+[KkMm]?)\s*\n\s*Views'),
    'reposts': re.compile(r'([\d,.]+[KkMm]?)\s*\n\s*Reposts'),
    'quotes': re.compile(r'([\d,.]+[KkMm]?)\s*\n\s*Quotes'),
    'likes': re.compile(r'([\d,.]+[KkMm]?)\s*\n\s*Likes'),
    'bookmarks': re.compile(r'([\d,.]+[KkMm]?)\s*\n\s*Bookmarks'),
}

def extract_engagement_metrics(text, patterns):
    """Extract engagement metrics from page text using regex patterns."""
    metrics = {}
    for key, pattern in patterns.items():
        match = pattern.search(text)
        metrics[key] = match.group(1) if match else None
    return metrics

def scrape_tweet(driver, tweet_url):
    """Scrape data from a single tweet URL."""
    driver.get(tweet_url)
    time.sleep(SCRAPE_DELAY_SECONDS)
    
    data = {}
    try:
        data['tweetDate'] = driver.find_element(By.XPATH, '//time').text
        data['username'] = driver.find_elements(By.CSS_SELECTOR, 'div a')[6].text
        data['text'] = driver.find_elements(By.CSS_SELECTOR, 'div span')[18].text
        
        full_text = driver.find_elements(By.CSS_SELECTOR, 'div div')[0].text
        data.update(extract_engagement_metrics(full_text, ENGAGEMENT_PATTERNS))
        
        images = driver.find_elements(By.XPATH, '//img[@alt="Image"]')
        data['imageUrl'] = images[0].get_attribute('src') if images else None
        
    except Exception as e:
        print(f"Error scraping tweet: {e}")
        for key in ['tweetDate', 'username', 'text', 'imageUrl', 'views', 'reposts', 'quotes', 'likes', 'bookmarks']:
            data[key] = "TWEET-NOT-FOUND"
    
    return data

def collect_tweet_data(source_tweets, output_dir_images, output_file):
    """Collect data for all tweets in the source dataframe."""
    drive.mount('/content/drive')
    driver = create_chrome_driver()
    os.makedirs(output_dir_images, exist_ok=True)
    
    source_tweets['tweetUrl'] = source_tweets['tweetId'].apply(
        lambda tid: f"https://twitter.com/anyuser/status/{tid}"
    )
    
    results = []
    
    for i, row in source_tweets.iterrows():
        tweet_id = row['tweetId']
        print(f"Processing tweet ID: {tweet_id}")
        
        tweet_data = scrape_tweet(driver, row['tweetUrl'])
        tweet_data['tweetId'] = tweet_id
        tweet_data['count'] = row['count']
        tweet_data['tweetUrl'] = row['tweetUrl']
        
        if tweet_data.get('imageUrl') and tweet_data['imageUrl'] != "TWEET-NOT-FOUND":
            image_path = f"{output_dir_images}/image_{tweet_id}.jpg"
            urllib.request.urlretrieve(tweet_data['imageUrl'], image_path)
        
        results.append(tweet_data)
        
        df = pd.DataFrame(results)
        df.to_csv(output_file, index=False)
        
        if (i + 1) % BATCH_SIZE == 0:
            print(f"Pausing for {BATCH_PAUSE_SECONDS // 60} minutes after processing {BATCH_SIZE} tweets.")
            time.sleep(BATCH_PAUSE_SECONDS)
    
    driver.quit()
    return df

In [ ]:
source_tweets = community_notes['visual']['tweetId'].value_counts().reset_index()
source_tweets.columns = ['tweetId', 'count']

hydrated_df = collect_tweet_data(source_tweets, OUTPUT_DIR_IMAGES, OUTPUT_FILE_INTERMEDIATE)

### 4.3 Follower Count Collection

In [ ]:
def collect_follower_counts(df, output_file):
    """Collect follower counts for each user in the dataframe."""
    driver = create_chrome_driver()
    df['userFollowers'] = None
    
    for i, row in df.iterrows():
        username = row.get('username')
        
        if not username or username == 'TWEET-NOT-FOUND':
            print("Skipping row with missing username.")
            df.at[i, 'userFollowers'] = 'TWEET-NOT-FOUND'
            continue
        
        username = username.lstrip('@')
        url = f"https://twitter.com/{username}"
        print(f"Collecting followers for: {username}")
        
        driver.get(url)
        time.sleep(SCRAPE_DELAY_SECONDS)
        
        try:
            span_elements = driver.find_elements(By.CSS_SELECTOR, 'span span')
            for j, element in enumerate(span_elements):
                if element.text == 'Followers':
                    df.at[i, 'userFollowers'] = span_elements[j - 1].text
                    print(f"Followers for {username}: {df.at[i, 'userFollowers']}")
                    break
        except Exception as e:
            print(f"Error collecting followers for {username}: {e}")
        
        if (i + 1) % BATCH_SIZE == 0:
            print("Saving intermediate data.")
            df.to_csv(output_file, index=False)
            print(f"Pausing for {BATCH_PAUSE_SECONDS // 60} minutes.")
            time.sleep(BATCH_PAUSE_SECONDS)
    
    driver.quit()
    df.to_csv(output_file, index=False)
    return df

In [ ]:
hydrated_df = collect_follower_counts(hydrated_df, OUTPUT_FILE_USERS)

### 4.4 Data Cleaning

Standardize numerical formats and transform dates into DD/MM/YYYY format.

In [ ]:
def format_number(value):
    """Convert abbreviated numbers (1.2M, 5K) to formatted strings."""
    if pd.isna(value) or value == '':
        return '0'
    
    try:
        text = str(value).upper()
        num = float(text.replace('M', '').replace('K', '').replace(',', ''))
        
        if 'M' in text:
            num *= 1e6
        elif 'K' in text:
            num *= 1e3
        
        if num >= 1e6:
            return f"{num/1e6:.1f}m".rstrip('0').rstrip('.')
        elif num >= 1e3:
            return f"{num/1e3:.1f}k".rstrip('0').rstrip('.')
        else:
            return str(int(num))
    except (ValueError, AttributeError):
        return str(value)

def parse_tweet_date(date_str, fallback_year="2023"):
    """Parse various tweet date formats into DD/MM/YYYY."""
    if not isinstance(date_str, str):
        return None
    
    date_formats = [
        '%I:%M %p \u00b7 %b %d, %Y',
        '%b %d, %Y',
    ]
    
    for fmt in date_formats:
        try:
            return datetime.strptime(date_str, fmt).strftime('%d/%m/%Y')
        except ValueError:
            continue
    
    try:
        return datetime.strptime(f"{date_str}, {fallback_year}", '%b %d, %Y').strftime('%d/%m/%Y')
    except ValueError:
        return None

def clean_dataframe(df):
    """Clean and standardize the tweet dataframe."""
    numeric_columns = ['views', 'reposts', 'quotes', 'likes', 'bookmarks', 'userFollowers']
    
    for col in numeric_columns:
        df[col] = df[col].astype(str).apply(format_number)
    
    df['tweetDate'] = df['tweetDate'].apply(parse_tweet_date)
    df = df.rename(columns={'count': 'notesCount'})
    
    return df

In [ ]:
processed_df = clean_dataframe(hydrated_df)
processed_df.to_csv(OUTPUT_FILE_FINAL, index=False)

---

# Data Annotation Codebook

## 1. Introduction

This codebook provides guidance for manual coding of the X dataframe obtained from Community Notes data. The dataset contains 16 columns:

| Column | Description |
|--------|-------------|
| tweetId | Unique identifier for each tweet |
| tweetDate | Date when the tweet was posted |
| username | Username of the account that posted the tweet |
| userFollowers | Number of followers of the user |
| verified | Whether the user account is verified (TRUE/FALSE) |
| noteCount | Total count of community notes for the tweet |
| tweetUrl | URL directing to the tweet |
| imageUrl | URL of the accompanying image |
| views | Number of times the tweet has been viewed |
| reposts | Number of times the tweet has been reposted |
| quotes | Number of times the tweet has been quoted |
| likes | Number of likes the tweet has received |
| bookmarks | Number of times the tweet has been bookmarked |
| political | Label indicating if the tweet is POLITICAL or NON-POLITICAL |
| media | Type of media attached (IMAGE or VIDEO) |
| text | The text content of the tweet |

## 2. Variables and Coding Instructions

### Tweet Status

- **REMOVED**: Tweet is no longer accessible
- **NO-DATA**: Tweet lacks impression data (posted before feature rollout in Nov-Dec 2022)
- **NOT-AI-GEN**: Media is not AI-generated per community notes consensus

### Verified Status

- **TRUE**: User has a verification tick (any color, including organization verification)
- **FALSE**: User does not have a verification tick

### Political Status

- **POLITICAL**: Tweets explicitly or implicitly referencing politicians, governments, political parties, or political issues
- **NON-POLITICAL**: Tweets not meeting the above criteria

## 3. Labelling Workflow

1. **Initial Inspection**: Review each row to assess tweet status. Mark as REMOVED if inaccessible, NO-DATA if missing impressions.

2. **AI Detection**: Inspect accompanying images. Mark as NOT-AI-GEN if media is not AI-generated per community notes consensus.

3. **Verification Check**: Examine user profile for verification tick. Update verified column with TRUE or FALSE.

4. **Political Classification**: Assess tweet content and update political column with POLITICAL or NON-POLITICAL.

## 4. Inter-Rater Reliability

We employ **Krippendorff's Alpha** to measure agreement among raters. This method evaluates reliability across different measurement levels, suitable for datasets with diverse variables.